## Name: Asit Jain
## Roll No: M25DE1049
## Assignment 3 - MLBD

# Part 6: Explainability in Recommender Systems
## Task 11: Neighborhood-Based Explanations (For Collaborative Filtering)

Show which similar users or items influenced a recommendation.

Example: *"Users who liked Inception also liked Interstellar."*

**Approach:** k-Nearest Neighbors explanations based on user and item similarity.

**Dataset:** MovieLens ml-latest-small

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'pandas', 'numpy', 'scikit-learn', 'scipy', 'matplotlib', '-q'])

import os, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
print('All imports successful!')

### Step 1: Load Data and Build User-Item Matrix

In [ ]:
url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
zip_path = 'ml-latest-small.zip'
extract_dir = 'ml-latest-small'
if not os.path.exists(extract_dir):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z: z.extractall('.')

movies = pd.read_csv(os.path.join(extract_dir, 'movies.csv'))
ratings = pd.read_csv(os.path.join(extract_dir, 'ratings.csv'))

# Build user-item matrix
user_ids = sorted(ratings['userId'].unique())
movie_ids = sorted(ratings['movieId'].unique())
uid_map = {u: i for i, u in enumerate(user_ids)}
mid_map = {m: i for i, m in enumerate(movie_ids)}
idx_to_uid = {i: u for u, i in uid_map.items()}
idx_to_mid = {i: m for m, i in mid_map.items()}

rows = ratings['userId'].map(uid_map).values
cols = ratings['movieId'].map(mid_map).values
vals = ratings['rating'].values
ui_matrix = csr_matrix((vals, (rows, cols)), shape=(len(user_ids), len(movie_ids)))
ui_dense = ui_matrix.toarray()

# Movie title lookup
mid_to_title = dict(zip(movies['movieId'], movies['title']))
print(f'User-Item matrix: {ui_dense.shape}')

### Step 2: Compute User and Item Similarity

In [ ]:
# User similarity (cosine)
user_sim = cosine_similarity(ui_dense)
np.fill_diagonal(user_sim, 0)

# Item similarity (cosine)
item_sim = cosine_similarity(ui_dense.T)
np.fill_diagonal(item_sim, 0)

print(f'User similarity matrix: {user_sim.shape}')
print(f'Item similarity matrix: {item_sim.shape}')

### Step 3: User-Based Neighborhood Explanation

For a predicted rating, show which similar users influenced the prediction and what they rated.

In [ ]:
def explain_user_cf(user_id, movie_id, k=10):
    """Explain a User-CF recommendation by showing influential neighbors."""
    if user_id not in uid_map or movie_id not in mid_map:
        print('User or movie not found.')
        return None

    uid_idx = uid_map[user_id]
    mid_idx = mid_map[movie_id]
    title = mid_to_title.get(movie_id, 'Unknown')

    # Find k most similar users who rated this movie
    sims = user_sim[uid_idx]
    raters = np.where(ui_dense[:, mid_idx] > 0)[0]
    if len(raters) == 0:
        print(f'No users rated "{title}"')
        return None

    rater_sims = [(r, sims[r], ui_dense[r, mid_idx]) for r in raters if sims[r] > 0]
    rater_sims.sort(key=lambda x: x[1], reverse=True)
    top_k = rater_sims[:k]

    if not top_k:
        print('No similar users found.')
        return None

    # Weighted prediction
    weights = np.array([s for _, s, _ in top_k])
    ratings_arr = np.array([r for _, _, r in top_k])
    pred = np.average(ratings_arr, weights=weights)

    print(f'Movie: "{title}"')
    print(f'Predicted rating for User {user_id}: {pred:.2f}')
    print(f'\nInfluential similar users (k={len(top_k)}):')
    print(f'{"User":>6s} | {"Similarity":>10s} | {"Their Rating":>12s}')
    print('-' * 35)
    for r_idx, sim, rating in top_k:
        print(f'{idx_to_uid[r_idx]:>6d} | {sim:>10.4f} | {rating:>12.1f}')

    # Natural language explanation
    avg_neighbor_rating = np.mean(ratings_arr)
    print(f'\n💡 {len(top_k)} users similar to you rated "{title}" an average of {avg_neighbor_rating:.1f}★.')
    if avg_neighbor_rating >= 4.0:
        print(f'   Users with similar taste really enjoyed this movie.')
    elif avg_neighbor_rating >= 3.0:
        print(f'   Users with similar taste found this movie decent.')
    else:
        print(f'   Users with similar taste did not rate this highly.')

    return {'pred': pred, 'neighbors': top_k}

# Example explanations
print('='*65)
result = explain_user_cf(1, 1, k=5)  # User 1, Toy Story
print()
print('='*65)
result = explain_user_cf(1, 296, k=5)  # User 1, Pulp Fiction

### Step 4: Item-Based Neighborhood Explanation

Show which items the user already liked that are similar to the recommended item.

In [ ]:
def explain_item_cf(user_id, movie_id, k=5):
    """Explain an Item-CF recommendation by showing similar items the user liked."""
    if user_id not in uid_map or movie_id not in mid_map:
        print('User or movie not found.')
        return None

    uid_idx = uid_map[user_id]
    mid_idx = mid_map[movie_id]
    title = mid_to_title.get(movie_id, 'Unknown')

    # Find items the user rated
    user_ratings = ui_dense[uid_idx]
    rated_items = np.where(user_ratings > 0)[0]
    if len(rated_items) == 0:
        print('User has no ratings.')
        return None

    # Get similarity between target movie and user's rated movies
    sims = item_sim[mid_idx]
    item_data = [(idx, sims[idx], user_ratings[idx]) for idx in rated_items if sims[idx] > 0]
    item_data.sort(key=lambda x: x[1], reverse=True)
    top_k = item_data[:k]

    if not top_k:
        print('No similar items found.')
        return None

    # Weighted prediction
    weights = np.array([s for _, s, _ in top_k])
    ratings_arr = np.array([r for _, _, r in top_k])
    pred = np.average(ratings_arr, weights=weights)

    print(f'Target Movie: "{title}"')
    print(f'Predicted rating for User {user_id}: {pred:.2f}')
    print(f'\nBecause you liked these similar movies:')
    print(f'{"Movie":40s} | {"Similarity":>10s} | {"Your Rating":>11s}')
    print('-' * 68)
    for i_idx, sim, rating in top_k:
        m_id = idx_to_mid[i_idx]
        t = mid_to_title.get(m_id, 'Unknown')[:38]
        print(f'{t:40s} | {sim:>10.4f} | {rating:>11.1f}')

    # Natural language
    similar_titles = [mid_to_title.get(idx_to_mid[i], '')[:30] for i, _, _ in top_k[:3]]
    print(f'\n💡 "{title}" is recommended because you enjoyed {", ".join(similar_titles)}.')

    return {'pred': pred, 'similar_items': top_k}

# Example explanations
print('='*70)
result = explain_item_cf(1, 2571, k=5)  # User 1, Matrix
print()
print('='*70)
result = explain_item_cf(5, 356, k=5)  # User 5, Forrest Gump

### Step 5: Visualize Neighbor Influence

In [ ]:
def visualize_item_explanation(user_id, movie_id, k=8):
    """Visualize item-based explanation as a bar chart."""
    uid_idx = uid_map[user_id]
    mid_idx = mid_map[movie_id]
    title = mid_to_title.get(movie_id, 'Unknown')
    user_ratings = ui_dense[uid_idx]
    rated_items = np.where(user_ratings > 0)[0]
    sims = item_sim[mid_idx]
    item_data = [(idx, sims[idx], user_ratings[idx]) for idx in rated_items if sims[idx] > 0]
    item_data.sort(key=lambda x: x[1], reverse=True)
    top_k = item_data[:k]

    if not top_k:
        print('No similar items to visualize.')
        return

    names = [mid_to_title.get(idx_to_mid[i], '?')[:25] for i, _, _ in top_k]
    similarities = [s for _, s, _ in top_k]
    user_rats = [r for _, _, r in top_k]
    contributions = [s * r for s, r in zip(similarities, user_rats)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Similarity bars
    colors = ['#4CAF50' if r >= 4 else '#FF9800' if r >= 3 else '#F44336' for r in user_rats]
    axes[0].barh(names[::-1], similarities[::-1], color=colors[::-1], edgecolor='black', alpha=0.8)
    axes[0].set_xlabel('Cosine Similarity')
    axes[0].set_title(f'Similar Movies to "{title[:30]}"')

    # Contribution bars (similarity × rating)
    axes[1].barh(names[::-1], contributions[::-1], color=colors[::-1], edgecolor='black', alpha=0.8)
    axes[1].set_xlabel('Contribution (Similarity × Your Rating)')
    axes[1].set_title('Influence on Prediction')

    plt.tight_layout()
    plt.show()

visualize_item_explanation(1, 2571, k=8)  # User 1, Matrix

### Step 6: Compare User-CF vs Item-CF Explanations for Same Prediction

In [ ]:
# Compare both explanation types for the same recommendation
test_cases = [(1, 260), (5, 1196), (10, 480)]  # Various (user, movie) pairs

for uid, mid in test_cases:
    title = mid_to_title.get(mid, 'Unknown')
    print('='*70)
    print(f'EXPLAINING: User {uid} → "{title}"')
    print('='*70)
    print('\n--- User-Based Explanation ---')
    explain_user_cf(uid, mid, k=3)
    print('\n--- Item-Based Explanation ---')
    explain_item_cf(uid, mid, k=3)
    print()

### Step 7: Explanation Coverage Analysis

How often can we provide meaningful explanations?

In [ ]:
# Check explanation coverage: what % of predictions can be explained?
np.random.seed(42)
sample_users = np.random.choice(user_ids, size=100, replace=False)
user_cf_explainable = 0
item_cf_explainable = 0
total_pairs = 0

for uid in sample_users:
    uid_idx = uid_map[uid]
    rated = np.where(ui_dense[uid_idx] > 0)[0]
    # Pick 5 random unrated movies
    unrated = np.where(ui_dense[uid_idx] == 0)[0]
    if len(unrated) == 0:
        continue
    sample_movies = np.random.choice(unrated, size=min(5, len(unrated)), replace=False)

    for mid_idx in sample_movies:
        total_pairs += 1
        # User-CF: any similar user rated this?
        sims = user_sim[uid_idx]
        raters = np.where(ui_dense[:, mid_idx] > 0)[0]
        if any(sims[r] > 0 for r in raters):
            user_cf_explainable += 1
        # Item-CF: any similar item rated by user?
        isims = item_sim[mid_idx]
        if any(isims[r] > 0 for r in rated):
            item_cf_explainable += 1

print(f'Explanation Coverage (out of {total_pairs} test pairs):')
print(f'  User-CF: {user_cf_explainable}/{total_pairs} ({user_cf_explainable/total_pairs*100:.1f}%)')
print(f'  Item-CF: {item_cf_explainable}/{total_pairs} ({item_cf_explainable/total_pairs*100:.1f}%)')

### Analysis

**Key Findings:**

1. **User-CF explanations** answer *"who else liked this?"* — they show that people with similar taste enjoyed the movie. This builds social proof but can feel abstract since users don't know the neighbors.

2. **Item-CF explanations** answer *"what similar movies did I enjoy?"* — they connect the recommendation to the user's own history. This is more intuitive and personal (e.g., *"Because you liked The Matrix and Inception"*).

3. **Coverage**: Item-CF typically has higher explanation coverage because users have rated many items, providing more potential similar-item connections. User-CF depends on finding neighbors who rated the specific target movie.

4. **Complementary approaches**: User-CF explanations work well for popular movies (many raters), while Item-CF works well for users with rich histories. Combining both gives the most complete picture.